In [ ]:
import os
import shutil
import pandas as pd
import kagglehub

# Download latest version
path = kagglehub.dataset_download("emirhanai/social-media-usage-and-emotional-well-being")

print("Path to dataset files:", path)

# Define the source directory (where KaggleHub stored the dataset)
source_dir = "/root/.cache/kagglehub/datasets/emirhanai/social-media-usage-and-emotional-well-being/versions/1"

# Define the target directory in session storage
target_dir = "/content/dataset"

# Create the target directory if it doesn't exist
os.makedirs(target_dir, exist_ok=True)

# Copy all files from the source directory to the target directory
for filename in os.listdir(source_dir):
    src_file = os.path.join(source_dir, filename)
    dst_file = os.path.join(target_dir, filename)
    if os.path.isfile(src_file):
        shutil.copy(src_file, dst_file)

print("Files in /content/dataset:")
print(os.listdir(target_dir))


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix, roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Define file paths
train_file = "/content/dataset/train.csv"
val_file = "/content/dataset/val.csv"

# Load train.csv and val.csv
df_train = pd.read_csv(train_file)
# Use on_bad_lines="skip" to avoid parser errors in val.csv
df_val = pd.read_csv(val_file, on_bad_lines="skip")

print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)


In [ ]:
# Combine train and validation data
df_combined = pd.concat([df_train, df_val], ignore_index=True)
print("Combined data shape:", df_combined.shape)

In [ ]:
df_combined.head()

In [ ]:
df_combined.info()

In [ ]:
df_combined.describe()

In [ ]:
df_combined.isnull().sum()

In [ ]:
# Drop rows with any null values
df_combined.dropna(inplace=True)
print("Shape after dropping nulls:", df_combined.shape)

# Convert 'Age' to numeric (non-numeric values become NaN)
df_combined['Age'] = pd.to_numeric(df_combined['Age'], errors='coerce')
# Drop rows where 'Age' conversion failed
df_combined = df_combined.dropna(subset=['Age'])
print("Shape after cleaning Age:", df_combined.shape)

# Drop the 'User_ID' column (if present) since it is an identifier
if 'User_ID' in df_combined.columns:
    df_combined.drop('User_ID', axis=1, inplace=True)


In [ ]:
# List of numeric columns
numeric_cols = [
    "Age",
    "Daily_Usage_Time (minutes)",
    "Posts_Per_Day",
    "Likes_Received_Per_Day",
    "Comments_Received_Per_Day",
    "Messages_Sent_Per_Day"
]

df_combined[numeric_cols].hist(figsize=(15, 10), bins=20)
plt.suptitle("Histograms for Numeric Columns", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
plt.figure(figsize=(15, 10))
for i, col in enumerate(numeric_cols):
    plt.subplot(2, 3, i + 1)
    sns.boxplot(y=df_combined[col])
    plt.title(f"Boxplot of {col}")
plt.tight_layout()
plt.show()


In [ ]:
# Count plot for Dominant_Emotion
plt.figure(figsize=(8, 6))
sns.countplot(x="Dominant_Emotion", data=df_combined, palette="viridis")
plt.title("Count of Dominant Emotions")
plt.xlabel("Dominant Emotion")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(x="Gender", data=df_combined, palette="coolwarm")
plt.title("Count of Genders")
plt.xlabel("Gender")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(x="Platform", data=df_combined, palette="pastel")
plt.title("Count of Platforms")
plt.xlabel("Platform")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
corr_matrix = df_combined[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Heatmap for Numeric Variables")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    x="Age",
    y="Daily_Usage_Time (minutes)",
    hue="Dominant_Emotion",
    data=df_combined,
    palette="bright"
)
plt.title("Age vs Daily Usage Time by Dominant Emotion")
plt.xlabel("Age")
plt.ylabel("Daily Usage Time (minutes)")
plt.legend(title="Dominant Emotion", loc="upper right")
plt.show()

In [ ]:
# Encode the target variable 'Dominant_Emotion' into numerical labels
label_encoder = LabelEncoder()
df_combined['Dominant_Emotion'] = label_encoder.fit_transform(df_combined['Dominant_Emotion'])

# Separate features (X) and target (y)
X = df_combined.drop('Dominant_Emotion', axis=1)
y = df_combined['Dominant_Emotion']

# One-hot encode categorical features ('Gender' and 'Platform')
X = pd.get_dummies(X, columns=['Gender', 'Platform'], drop_first=True)

print("\nFeatures after one-hot encoding:")
X.head()

In [ ]:
# Split the combined data into training and testing sets (e.g., 80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Initialize and train a Random Forest Classifier
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("\nModel Accuracy on test set:", accuracy)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Get predicted probabilities
y_score = model.predict_proba(X_test)

# Use the classes from the model
n_classes = len(model.classes_)
print("Number of classes (from model.classes_):", n_classes)

# Binarize y_test using the model's classes
y_test_binarized = label_binarize(y_test, classes=model.classes_)

# Compute ROC curve and ROC area for each class
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_binarized[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Compute micro-average ROC curve and ROC area
fpr["micro"], tpr["micro"], _ = roc_curve(y_test_binarized.ravel(), y_score.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# Plot ROC curves for each class
plt.figure(figsize=(10,8))
colors = plt.cm.get_cmap('Set1', n_classes)

for i in range(n_classes):
    plt.plot(fpr[i], tpr[i], color=colors(i), lw=2,
             label='ROC curve for class {0} (AUC = {1:0.2f})'.format(model.classes_[i], roc_auc[i]))

# Plot micro-average ROC curve
plt.plot(fpr["micro"], tpr["micro"], color='deeppink', linestyle=':', linewidth=4,
         label='Micro-average ROC (AUC = {0:0.2f})'.format(roc_auc["micro"]))

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Multi-class Classification')
plt.legend(loc="lower right")
plt.show()

In [ ]:
importances = model.feature_importances_
features = X.columns

plt.figure(figsize=(10, 6))
sns.barplot(x=importances, y=features)
plt.title("Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()